### Unificación de datos

El siguiente notebook se encarga de mezclar todos los archivos csv de datos de buses y metro, para crear dos nuevos archivos csv con estos datos unificados.

In [83]:
import pandas as pd
import pandas.errors
from os import listdir
from os.path import join

buses_csvs = listdir(join("..", "buses_outputs"))
metro_csvs = listdir(join("..", "metro_outputs"))

if ".ipynb_checkpoints" in buses_csvs:
    buses_csvs.remove(".ipynb_checkpoints")
if ".ipynb_checkpoints" in metro_csvs:
    metro_csvs.remove(".ipynb_checkpoints")

unified_bus_df_list = []
unified_metro_df_list = []

for bus_csv in buses_csvs:
    bus_csv_df = pd.read_csv(join("..", "buses_outputs", bus_csv))
    unified_bus_df_list.append(bus_csv_df)

for metro_csv in metro_csvs: # Habemus un problemón los datos desde el 25-10-25 están bugueados
    try:
        metro_csv_df = pd.read_csv(join("..", "metro_outputs", metro_csv))
    except pandas.errors.EmptyDataError:
        continue
    unified_metro_df_list.append(metro_csv_df)

unified_bus_df = pd.concat(unified_bus_df_list, ignore_index=True, sort=False)
unified_metro_df = pd.concat(unified_metro_df_list, ignore_index=True, sort=False)
unified_bus_df.drop(columns=["X", "Y"], inplace=True)

Añadiremos datos de cada paradero al dataframe.

In [84]:
import json
# Abrimos el archivo donde tenemos información importantes de los pasajeros
ruta_json = join("..", "data", "Paraderos-Santiago-Chile.geojsonl.json")
# Guardamos su informacion en una lista
datos = []
with open(ruta_json, "r", encoding="utf-8-sig") as f:
    for archivo in f:
        datos.append(json.loads(archivo.strip()))
# creamos un datafraem con los datos de la lista
df_paradero = pd.json_normalize(datos)
# Renombramos columnas que estarán en nuestro DataFrame
df_paradero = df_paradero.rename(columns={"properties.ID": "id", 
                            "properties.CODINFRA": "codinfra",
                            "properties.SIMT": "bus_stop_code",
                            "properties.COMUNA": "comuna",
                            "properties.NOMBRE_PAR": "nombre_par",
                            "properties.NSERVICIOS": "n_servicios",
                            "properties.SERVICIOS": "servicios",
                            "geometry.coordinates": "coordinates"})
# Seleccionamos dichas columnas
df_paradero = df_paradero[["id", "codinfra", "comuna", "bus_stop_code", "nombre_par", "n_servicios", "servicios", "coordinates"]]
# Reemplazamos nombres de comunas pues el archivo venía con un encoding incorrecto
df_paradero["comuna"] = df_paradero["comuna"].replace(
    {"CONCHAL�": "CONCHALÍ",
    "ESTACI�N CENTRAL": "ESTACIÓN CENTRAL",
    "MAIP�": "MAIPÚ",
    "PE�ALOL�N": "PEÑALOLÉN",
    "SAN JOAQU�N": "SAN JOAQUÍN",
    "SAN RAM�N": "SAN RAMÓN",
    "�U�OA": "ÑUÑOA"
    })
df_paradero.head()

,id,codinfra,comuna,bus_stop_code,nombre_par,n_servicios,servicios,coordinates
0,11672,L-35-6-5-OP,PADRE HURTADO,PI1923,Cam. Bajos de Sta. Cruz / esq. El Muelle,1,I35I,"[-70.823088, -33.572525]"
1,11671,L-35-1-67-OP,PADRE HURTADO,PI1935,Cam. Sn. Alberto Hurtado / esq. Sn. Fco de Borja,1,I35I,"[-70.8189, -33.571649]"
2,11674,L-35-5-48-NS,PADRE HURTADO,PI1926,Avenida San Ignacio / esq. Cam. Sn. A. Hurtado,1,I35I,"[-70.818197, -33.57202]"
3,11693,L-35-5-49-SN,PADRE HURTADO,PI1925,Avenida San Ignacio / esq. Cam. Sn. A. Hurtado,1,I35R,"[-70.817967, -33.57187]"
4,11675,L-35-5-54-NS,PADRE HURTADO,PI1927,Avenida San Ignacio / esq. San Alberto,1,I35I,"[-70.816035, -33.574243]"


Reparamos las coordenadas de cada paradero.

In [85]:
unified_bus_df = pd.merge(unified_bus_df, df_paradero, on= "bus_stop_code", how="inner")
unified_bus_df[["lan", "lon"]] = pd.DataFrame(unified_bus_df["coordinates"].tolist(), index=unified_bus_df.index)
unified_bus_df.drop(columns=["coordinates"], inplace= True)
unified_bus_df.head()

,bus_stop_code,route_id,bus_id,meters_distance,min_arrival_time,max_arrival_time,date,id,codinfra,comuna,nombre_par,n_servicios,servicios,lan,lon
0,PB1013,B12,SKDG-90,7520,16,24,2025-10-12 19:01:28.530087,8983,L-6-39-30-OP,QUILICURA,San Ignacio / esq. Galvarino,3,B12I;B12cR;B33I,-70.705935,-33.331142
1,PB1013,B33,SKPP-55,2951,8,10,2025-10-12 19:01:28.530094,8983,L-6-39-30-OP,QUILICURA,San Ignacio / esq. Galvarino,3,B12I;B12cR;B33I,-70.705935,-33.331142
2,PI725,I10,FLXK-93,2315,7,9,2025-10-12 19:01:30.454781,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02I;I10R;I10NR,-70.751076,-33.496300
3,PI725,I10,FLXJ-25,6384,18,26,2025-10-12 19:01:30.454787,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02I;I10R;I10NR,-70.751076,-33.496300
4,PI725,I02,FLXY-79,4575,12,16,2025-10-12 19:01:30.454789,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02I;I10R;I10NR,-70.751076,-33.496300


Podemos notar al revisar este DataFrame, en la columna 'servicios', que todos los códigos de micro tienen una letra al final (Sabemos que puede ser N, C, E, V) pero también las hay en casos que no corresponden. En consecuencia, es necesario que los datos correspondientes a los recorridos sean los correctos-

In [86]:
clean_unified_bus_df = unified_bus_df.copy()
clean_unified_bus_df["servicios"] = clean_unified_bus_df["servicios"].str.replace(r"([IR])(?=;|$)", "", regex=True)
clean_unified_bus_df.head()

,bus_stop_code,route_id,bus_id,meters_distance,min_arrival_time,max_arrival_time,date,id,codinfra,comuna,nombre_par,n_servicios,servicios,lan,lon
0,PB1013,B12,SKDG-90,7520,16,24,2025-10-12 19:01:28.530087,8983,L-6-39-30-OP,QUILICURA,San Ignacio / esq. Galvarino,3,B12;B12c;B33,-70.705935,-33.331142
1,PB1013,B33,SKPP-55,2951,8,10,2025-10-12 19:01:28.530094,8983,L-6-39-30-OP,QUILICURA,San Ignacio / esq. Galvarino,3,B12;B12c;B33,-70.705935,-33.331142
2,PI725,I10,FLXK-93,2315,7,9,2025-10-12 19:01:30.454781,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02;I10;I10N,-70.751076,-33.496300
3,PI725,I10,FLXJ-25,6384,18,26,2025-10-12 19:01:30.454787,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02;I10;I10N,-70.751076,-33.496300
4,PI725,I02,FLXY-79,4575,12,16,2025-10-12 19:01:30.454789,4883,L-13-68-5-OP,MAIPÚ,Rafael Riesco / esq. 1� Transversal,3,I02;I10;I10N,-70.751076,-33.496300


Por último, exportamos los dataframes en csv a la carpeta csv_unificado_{servicio}.

In [87]:
clean_unified_bus_df.to_csv(join("..", "csv_unificado_buses", "unified_bus_data.csv"), index=False)
unified_metro_df.to_csv(join("..", "csv_unificado_metro", "unified_metro_data.csv"), index=False)